
# 練習問題: 本当にGPUで実行されているか確かめる

## 目標

`#pragma omp target` (Fortran では `!$omp target` ... `!$omp end target`) は, GPUが使えないときには **黙ってCPUにフォールバック** して実行される.
そのため「GPUで動かしているつもりが, 実はCPUで動いていた」という事故が起こりうる.

そこで `omp_is_initial_device()` を使って, ある領域が実際にどこで実行されたかを判定する方法と, 環境変数 `OMP_TARGET_OFFLOAD` で実行場所を制御する方法を体験する.

- `omp_is_initial_device()` は, **ホスト(CPU)上では 1**, **デバイス(GPU)上では 0** を返す.

## 課題

`where_am_i.cpp` (または `where_am_i.f90`) は, まずホスト上で `omp_is_initial_device()` の値を表示し, 続いて `target` 領域の中で同じ関数の値を表示する.
コメント `TODO` の指示に従って **OpenMP の指示行を1つ追加** し, 後半の表示が `target` 領域(GPU上)で実行されるようにせよ.

- C++: メッセージを表示するブロック `{ ... }` の直前に `#pragma omp target` を1行加える.
- Fortran: `print` 文を `!$omp target` と `!$omp end target` で囲む.

表示するだけなので `map` 節を考える必要はない. それ以外のコードを変更する必要はない.

## コンパイルと実行

```
# C++
nvc++ -mp=gpu where_am_i.cpp -o where_am_i.exe

# Fortran
nvfortran -mp=gpu where_am_i.f90 -o where_am_i.exe
```

GPUは計算ノードにのみ搭載されているので, `%%bash_submit` でジョブとして投入して実行する.
以下の **3通り** の方法で実行し, `inside target:` の行に表示される値を比べよ.

```
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:10:00

# (1) 既定の動作 (GPUがあればGPU, なければCPU)
./where_am_i.exe

# (2) GPUでの実行を強制 (GPUが使えなければエラー)
OMP_TARGET_OFFLOAD=MANDATORY ./where_am_i.exe

# (3) 必ずCPUで実行
OMP_TARGET_OFFLOAD=DISABLED ./where_am_i.exe
```

## 期待される結果

- `inside target:` の値が **0** なら, その領域は **GPU** で実行されている.
- `inside target:` の値が **1** なら, その領域は **CPU** で実行されている (フォールバック).
- `on host:` の値は常に 1 (こちらは必ずホストで実行されるため).

計算ノードでの想定:

- (2) `MANDATORY` では `inside target:` は **0** になる (GPUが使われている).
- (3) `DISABLED` では `inside target:` は **1** になる (CPUに強制).
- (1) 既定では, 計算ノードにGPUがあれば **0** になるはず.

このように, `printf` / `print` の出力だけでは区別がつかない「どこで実行されたか」を, `omp_is_initial_device()` で確実に判定できる.
GPU向けのコードを書いたら, 思い込みで終わらせず実際にGPUで動いていることを確認する習慣をつけよう.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ where_am_i.cpp
#include <cstdio>
#include <omp.h>

int main() {
  printf("on host: omp_is_initial_device() = %d\n", omp_is_initial_device());
  // TODO: 下のブロックの直前に #pragma omp target を1行追加し, ブロックの中身をデバイス(GPU)上で実行させよ. (表示するだけなので map 節は不要)
  {
    printf("inside target: omp_is_initial_device() = %d\n", omp_is_initial_device());
  }
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=gpu where_am_i.cpp -o where_am_i_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./where_am_i_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ where_am_i.f90
program where_am_i
  use omp_lib
  print "(a,i0)", "on host: omp_is_initial_device() = ", omp_is_initial_device()
  ! TODO: 下の print を !$omp target ... !$omp end target で囲み, デバイス(GPU)上で実行させよ. (表示するだけなので map 節は不要)
  print "(a,i0)", "inside target: omp_is_initial_device() = ", omp_is_initial_device()
  ! TODO: 上で始めた target 領域を閉じる (!$omp end target).
end program where_am_i

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=gpu where_am_i.f90 -o where_am_i_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./where_am_i_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:where_am_i.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:where_am_i.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:where_am_i.cpp}